# Covariance

Each variable in a dataset has its own mean and variance — but those numbers say nothing about how two variables are **related**. Age and height rise together; age and daily naps move in opposite directions; age and test grades seem unrelated. The number that captures this is the **covariance**: center both variables, multiply the coordinates, and average the products. Same signs dominate → positive; opposite signs dominate → negative; a lawless cloud → near zero. It is based on the sixth video of Lesson 4 (Probability Distributions with Multiple Variables).

## 1. Introduction

Sometimes we want to know how two random variables are related to each other — understanding these relations helps build accurate models and make better decisions. Consider $X$ = the age of a child, together with three different companions:

- $Y_1$ = the child's **height** in inches,
- $Y_2$ = the child's **grade** on some test,
- $Y_3$ = the child's **naps per day**.

Ten children, three scatter plots, three visibly different patterns: an upward diagonal, a patternless cloud, and a downward diagonal. Means and variances of each variable exist but cannot tell these plots apart — a new summary is needed.

After completing this notebook you will be able to:

- Compute the covariance $\mathrm{Cov}(X, Y) = \frac{1}{n}\sum_i (x_i - \mu_x)(y_i - \mu_y)$ by hand and in code.
- Explain *why* the sign of centered-coordinate products captures the trend.
- Interpret positive, near-zero, and negative covariance (grow together / unrelated / move oppositely).
- Know where the plain covariance falls short (units and scale), setting up correlation.

## 2. Theory

### What means and variances miss

The three datasets share $\mu_x = 10.5$ (the same ten ages), and each has its own mean: heights $\mu_{y_1} = 60$, grades $\mu_{y_2} = 5$, naps $\mu_{y_3} = 3.7$. Those are the balance points of the scatter plots. The variances exist too ($\mathrm{Var}(X) = 9.17$; $39.56$, $9.78$, $7.57$ for the three $Y$'s). But something else is going on that none of these numbers capture:

- age vs **height** is almost a diagonal going *up* to the right;
- age vs **naps** goes *down* to the right;
- age vs **grades** is all over the place.

### Cooking up a formula

**Step 1 — center.** Subtract $\mu_x$ from the $x$-coordinates and $\mu_y$ from the $y$-coordinates so the balance point moves to the origin. (For visualization one can also divide by the standard deviations to make both variances 1 — a preview of correlation.)

**Step 2 — look at signs.** After centering:

- *height plot:* points with positive $x$ tend to have positive $y$, negative $x$ goes with negative $y$ → the coordinates **tend to share their sign**;
- *naps plot:* positive $x$ goes with negative $y$ and vice versa → **opposite signs**;
- *grades plot:* a land with no rules — any sign combination happens.

**Step 3 — multiply.** The product of coordinates turns sign patterns into numbers: same signs → positive product, opposite signs → negative product.

**Step 4 — average.** Add all the products: mostly-positive products give a positive sum (height), mostly-negative a negative sum (naps), a mix cancels toward zero (grades). That average is the **covariance**:

$$
\mathrm{Cov}(X, Y) = \frac{\sum_i (x_i - \mu_x)(y_i - \mu_y)}{n}
$$

("Sum of products of coordinates — *almost*: first center the data, then average.")

### Reading the sign

$$
\mathrm{Cov} > 0: \text{ one variable grows with the other} \qquad
\mathrm{Cov} \approx 0: \text{ no (linear) influence} \qquad
\mathrm{Cov} < 0: \text{ one grows, the other shrinks}
$$

In distribution language, the same quantity is $\mathrm{Cov}(X, Y) = E\big[(X - \mu_X)(Y - \mu_Y)\big]$ — note that $\mathrm{Cov}(X, X) = E[(X-\mu_X)^2]$ is just the variance: covariance generalizes variance to *pairs*.

> **Convention note:** like the variance, covariance comes in a divide-by-$n$ (population) and a divide-by-$(n{-}1)$ (sample) flavor. The video divides by $n$ for the covariances but quotes $(n{-}1)$-style variances (9.17 etc.); NumPy's `np.cov` uses $n-1$ unless you pass `ddof=0`. The sign — the message — is the same either way.

## 3. Worked Examples

The ten children (ages 6–15, one per age):

| age $x$ | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 |
|---|---|---|---|---|---|---|---|---|---|---|
| height $y_1$ | 50 | 52 | 55 | 57 | 60 | 62 | 64 | 65 | 67 | 68 |
| grade $y_2$ | 5 | 7 | 8 | 3 | 1 | 1 | 6 | 10 | 2 | 7 |
| naps $y_3$ | 8 | 7 | 6 | 5 | 4 | 3 | 2 | 1 | 1 | 0 |

### Example 1: age vs height — positive

Centered ages: $-4.5, \dots, 4.5$; centered heights: $-10, -8, -5, -3, 0, 2, 4, 5, 7, 8$. Products: $45, 28, 12.5, 4.5, 0, 1, 6, 12.5, 24.5, 36$ — all $\ge 0$, summing to $170$:

$$
\mathrm{Cov}(X, Y_1) = \frac{170}{10} = 17 > 0
$$

### Example 2: age vs naps — negative

Centered naps: $4.3, 3.3, 2.3, 1.3, 0.3, -0.7, -1.7, -2.7, -2.7, -3.7$. Every product is negative ($-19.35, -11.55, \dots, -16.65$), summing to $-74.5$:

$$
\mathrm{Cov}(X, Y_3) = \frac{-74.5}{10} = -7.45 < 0
$$

### Example 3: age vs grades — near zero

Products: $0, -7, -7.5, 3, 2, -2, 1.5, 12.5, -10.5, 9$ — positives and negatives cancel almost perfectly, summing to $1$:

$$
\mathrm{Cov}(X, Y_2) = \frac{1}{10} = 0.1 \approx 0
$$

Summary: grow-together pairs have covariance $17 > 0$; independent-looking pairs $0.1 \approx 0$; oppositely-moving pairs $-7.45 < 0$.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

age = np.array([6, 7, 8, 9, 10, 11, 12, 13, 14, 15])
height = np.array([50, 52, 55, 57, 60, 62, 64, 65, 67, 68])
grades = np.array([5, 7, 8, 3, 1, 1, 6, 10, 2, 7])
naps = np.array([8, 7, 6, 5, 4, 3, 2, 1, 1, 0])

print(f"means: age {age.mean()}, height {height.mean()}, "
      f"grades {grades.mean()}, naps {naps.mean()}")

### Covariance from the definition, showing every step

In [ ]:
def covariance_table(x, y, x_name, y_name):
    cx = x - x.mean()                       # step 1: center
    cy = y - y.mean()
    products = cx * cy                      # step 3: multiply coordinates
    table = pd.DataFrame({x_name: x, y_name: y,
                          "x - mu_x": cx, "y - mu_y": cy,
                          "product": products})
    cov = products.mean()                   # step 4: average
    return table, cov

table, cov = covariance_table(age, height, "age", "height")
print(table)
print(f"\nsum of products = {table['product'].sum()},  "
      f"Cov = {table['product'].sum():.0f}/10 = {cov}")

### All three covariances

In [ ]:
for y, name, video in [(height, "height", 17), (grades, "grades", 0.1),
                       (naps, "naps  ", -7.45)]:
    _, cov = covariance_table(age, y, "age", name)
    print(f"Cov(age, {name}) = {cov:>6.2f}   (video: {video})")
print()
print(f"np.cov with ddof=0 agrees: "
      f"{np.cov(age, height, ddof=0)[0, 1]:.1f}, "
      f"{np.cov(age, grades, ddof=0)[0, 1]:.1f}, "
      f"{np.cov(age, naps, ddof=0)[0, 1]:.2f}")
print(f"(np.cov default divides by n-1: {np.cov(age, height)[0, 1]:.2f} etc.)")

### Covariance generalizes variance

$\mathrm{Cov}(X, X)$ is the average of $(x_i - \mu_x)^2$ — the variance itself.

In [ ]:
_, cov_xx = covariance_table(age, age, "age", "age2")
print(f"Cov(age, age) = {cov_xx:.2f}")
print(f"Var(age)      = {age.var():.2f}   (population, /n)")
print(f"sample variance (/n-1): {age.var(ddof=1):.2f}   <- the video's 9.17")

### The sign logic, made visible in counts

In [ ]:
for y, name in [(height, "height"), (grades, "grades"), (naps, "naps")]:
    products = (age - age.mean()) * (y - y.mean())
    pos, neg = int((products > 0).sum()), int((products < 0).sum())
    print(f"age vs {name:<6}: {pos} positive products, {neg} negative"
          f"  -> Cov = {products.mean():>6.2f}")

## 5. Visualizations

### The three scatter plots with their balance points

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, y, name, cov_val in [(axes[0], height, "height (in)", 17),
                             (axes[1], grades, "grade", 0.1),
                             (axes[2], naps, "naps per day", -7.45)]:
    ax.scatter(age, y, s=60, color="tab:blue", zorder=3)
    ax.plot(age.mean(), y.mean(), "s", color="tab:red", markersize=10,
            label=f"mean ({age.mean()}, {y.mean()})")
    ax.set_title(f"age vs {name}:  Cov = {cov_val}")
    ax.set_xlabel("age x")
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.suptitle("Same ages, three different relationships")
plt.tight_layout()
plt.show()

### Centered data and the sign quadrants

After centering, the plane splits into four quadrants: products are positive in the shaded quadrants (same signs) and negative in the white ones. Watch where each dataset's points fall.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
for ax, y, name in [(axes[0], height, "height"), (axes[1], grades, "grades"),
                    (axes[2], naps, "naps")]:
    cx = age - age.mean()
    cy = y - y.mean()
    products = cx * cy
    lim_x, lim_y = 6, np.abs(cy).max() * 1.25

    ax.axhline(0, color="black", linewidth=1)
    ax.axvline(0, color="black", linewidth=1)
    ax.fill_between([0, lim_x], 0, lim_y, color="tab:green", alpha=0.12)
    ax.fill_between([-lim_x, 0], -lim_y, 0, color="tab:green", alpha=0.12)
    colors = ["tab:green" if p > 0 else ("tab:red" if p < 0 else "gray")
              for p in products]
    ax.scatter(cx, cy, s=60, c=colors, zorder=3)
    ax.set_title(f"{name}: sum of products = {products.sum():.1f}")
    ax.set_xlabel("x - mu_x")
    ax.set_ylabel("y - mu_y")
    ax.set_xlim(-lim_x, lim_x)
    ax.set_ylim(-lim_y, lim_y)
plt.suptitle("Green quadrants: positive products. Covariance = the average product.")
plt.tight_layout()
plt.show()

### Each point's contribution

The products themselves, bar by bar: height's bars all push up, naps' all push down, grades' fight each other and cancel.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True)
for ax, y, name in [(axes[0], height, "height"), (axes[1], grades, "grades"),
                    (axes[2], naps, "naps")]:
    products = (age - age.mean()) * (y - y.mean())
    colors = ["tab:green" if p >= 0 else "tab:red" for p in products]
    ax.bar(age, products, color=colors, edgecolor="black")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"{name}: mean of bars = Cov = {products.mean():.2f}")
    ax.set_xlabel("age")
    ax.grid(True, alpha=0.3, axis="y")
axes[0].set_ylabel("(x - mu_x)(y - mu_y)")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Without computing: what sign do you expect for the covariance of (hours of daylight, heating costs) across a year? And (study hours, exam score)? Verify your intuition on invented five-point datasets.

**Exercise 2.** Compute $\mathrm{Cov}(Y_3, X)$ — naps against age, in the reversed order. Compare with $\mathrm{Cov}(X, Y_3)$ and explain the (non-)difference.

### Intermediate

**Exercise 3.** Show numerically on the age/height data that $\mathrm{Cov}(X, Y) = E[XY] - E[X]E[Y]$ — the shortcut formula, the exact analogue of $\mathrm{Var}(X) = E[X^2] - E[X]^2$. Then derive it on paper with linearity.

**Exercise 4.** Scaling: convert heights to centimeters ($\times 2.54$) and recompute $\mathrm{Cov}(\text{age}, \text{height})$. Then add 100 to every height. Which operation changed the covariance, and following which rule? (This unit-dependence is covariance's big weakness.)

### Advanced

**Exercise 5.** Zero covariance does **not** imply "no relationship": construct the dataset $x = (-2, -1, 0, 1, 2)$, $y = x^2$, compute the covariance, and plot the points. What relationship does the covariance miss, and why does the sign logic fail here?

**Exercise 6.** Covariance from a joint PMF: for the (first die, second die) flat table and for the (first die, sum) band table, compute $\mathrm{Cov}(X, Y) = \sum_{x,y} (x - \mu_X)(y - \mu_Y) P_{XY}(x,y)$. Interpret the two answers (hint: for the sum, $\mathrm{Cov}(X, X + X_2) = \mathrm{Var}(X)$ when the dice are independent).

## 7. Solutions

In [ ]:
# Exercise 1: sign intuitions
daylight = np.array([8, 10, 12, 14, 16])
heating = np.array([90, 70, 50, 30, 20])
study = np.array([1, 2, 3, 4, 5])
score = np.array([55, 62, 70, 78, 90])

cov_dh = ((daylight - daylight.mean()) * (heating - heating.mean())).mean()
cov_ss = ((study - study.mean()) * (score - score.mean())).mean()
print(f"Exercise 1: Cov(daylight, heating) = {cov_dh:.1f} < 0 (more light, less heating)")
print(f"            Cov(study, score)     = {cov_ss:.1f} > 0 (more study, higher score)")

In [ ]:
# Exercise 2: symmetry
cov_xy = ((age - age.mean()) * (naps - naps.mean())).mean()
cov_yx = ((naps - naps.mean()) * (age - age.mean())).mean()
print(f"Exercise 2: Cov(age, naps) = {cov_xy:.2f}, Cov(naps, age) = {cov_yx:.2f}")
print("Identical: the product (x-mu_x)(y-mu_y) doesn't care about the order --")
print("covariance is symmetric, Cov(X, Y) = Cov(Y, X).")

In [ ]:
# Exercise 3: the shortcut formula
lhs = ((age - age.mean()) * (height - height.mean())).mean()
rhs = (age * height).mean() - age.mean() * height.mean()
print(f"Exercise 3: E[(X-mu)(Y-mu)] = {lhs:.2f},  E[XY] - E[X]E[Y] = {rhs:.2f}")
print("Paper derivation: expand (X - mu_X)(Y - mu_Y) = XY - mu_X Y - mu_Y X + mu_X mu_Y;")
print("take E, pull constants out: E[XY] - mu_X mu_Y - mu_Y mu_X + mu_X mu_Y")
print("= E[XY] - E[X]E[Y].")

In [ ]:
# Exercise 4: units change covariance
cov_inch = ((age - age.mean()) * (height - height.mean())).mean()
height_cm = height * 2.54
cov_cm = ((age - age.mean()) * (height_cm - height_cm.mean())).mean()
height_shift = height + 100
cov_shift = ((age - age.mean()) * (height_shift - height_shift.mean())).mean()

print(f"Exercise 4: Cov in inches: {cov_inch:.2f}")
print(f"Cov in cm:     {cov_cm:.2f}  = 2.54 * 17 (Cov(X, aY) = a Cov(X, Y))")
print(f"Cov shifted:   {cov_shift:.2f}  (adding a constant changes nothing --")
print("centering removes it). The value 17 vs 43.18 describes the SAME")
print("relationship in different units: covariance has units (years x inches),")
print("so its magnitude is hard to interpret. Fixing that is correlation's job.")

In [ ]:
# Exercise 5: zero covariance, perfect (nonlinear) relationship
x = np.array([-2, -1, 0, 1, 2])
y = x**2
cov = ((x - x.mean()) * (y - y.mean())).mean()

plt.figure(figsize=(5.5, 4))
plt.scatter(x, y, s=80, color="tab:blue", zorder=3)
plt.title(f"y = x^2: Cov = {cov:.1f}, yet y is a FUNCTION of x")
plt.xlabel("x"); plt.ylabel("y")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Exercise 5: Cov = {cov}")
print("The parabola is symmetric: every positive product on the right cancels a")
print("negative one on the left. Covariance detects LINEAR trends only -- zero")
print("covariance rules out a linear relationship, not any relationship.")

In [ ]:
# Exercise 6: covariance from joint PMFs
faces = np.arange(1, 7)
mu = 3.5

# flat table: independent dice
cov_flat = sum((x - mu) * (y - mu) * (1 / 36) for x in faces for y in faces)

# band table: X = first die, S = sum
cov_band = sum((x - mu) * ((x + y) - 7) * (1 / 36) for x in faces for y in faces)

print(f"Exercise 6: Cov(die1, die2)  = {cov_flat:.4f} (independent -> 0)")
print(f"            Cov(die1, sum)   = {cov_band:.4f} (= Var(die) = 35/12 = {35/12:.4f})")
print("The sum contains the first die, so they co-move exactly by Var(X):")
print("Cov(X, X + X2) = Cov(X, X) + Cov(X, X2) = Var(X) + 0.")

## 8. Conclusion

- Means and variances describe each variable alone; the **covariance** describes the *pair*: $\mathrm{Cov}(X, Y) = \frac{1}{n}\sum_i (x_i - \mu_x)(y_i - \mu_y)$ — center, multiply coordinates, average.
- The sign logic: same-sign coordinates (up-right/down-left) → positive products → $\mathrm{Cov} > 0$ (age & height: $17$); opposite signs → $\mathrm{Cov} < 0$ (age & naps: $-7.45$); a lawless cloud cancels → $\mathrm{Cov} \approx 0$ (age & grades: $0.1$).
- Useful identities: $\mathrm{Cov}(X, X) = \mathrm{Var}(X)$, symmetry, the shortcut $E[XY] - E[X]E[Y]$, and $\mathrm{Cov}(X, aY + b) = a\,\mathrm{Cov}(X, Y)$.
- Two caveats: covariance carries **units** (years × inches), so its magnitude is hard to compare across datasets — and it detects **linear** trends only (a perfect parabola scores 0).
- Both caveats are fixed by standardizing — the **correlation coefficient**, coming next.